# Train CatBoost (leak-free)
Rule: **split first, then fit anything.** The cleaning statistics (median/mode) and the feature selection
are learned from the training split only, then *applied* to the validation and test splits.
That keeps the test score an honest estimate of the Kaggle score.

In [1]:
import numpy as np
import pandas as pd

target = "transported"


def standardize_columns(df):
    """Strip, lowercase and snake_case column names (same as visual.ipynb)."""
    out = df.copy()
    out.columns = out.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
    return out


df = standardize_columns(pd.read_csv("./data/raw/train.csv"))
df.head()

,passengerid,homeplanet,cryosleep,cabin,destination,age,vip,roomservice,foodcourt,shoppingmall,spa,vrdeck,name,transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


## Split first: train : validation : test = 60 : 20 : 20
Done on the raw rows, before any statistic is computed.

In [2]:
from sklearn.model_selection import train_test_split

train_val_df, test_df = train_test_split(df, test_size=0.2, random_state=42,
                                         stratify=df[target])
train_df, val_df = train_test_split(train_val_df, test_size=0.25, random_state=42,
                                    stratify=train_val_df[target])

print(train_df.shape, val_df.shape, test_df.shape)

(5215, 14) (1739, 14) (1739, 14)


## Clean: fit on train, apply everywhere
`fit_cleaner` learns the fill values from one DataFrame; `apply_cleaner` uses them on any DataFrame
(validation, test, or Kaggle's `test.csv`). Rows are never dropped.

In [3]:
MONEY_COLS = ["roomservice", "foodcourt", "shoppingmall", "spa", "vrdeck"]
CATEGORY_COLS = ["homeplanet", "cryosleep", "destination", "vip"]
BOOL_COLS = ["cryosleep", "vip"]


def fit_cleaner(df):
    """Learn fill values from df (call this on training data only)."""
    stats = {"age": df["age"].median()}
    for col in CATEGORY_COLS:
        stats[col] = df[col].mode()[0]
    return stats


def apply_cleaner(df, stats):
    """Fill missing values with the learned stats; never drops rows."""
    out = df.copy()
    out["age"] = out["age"].fillna(stats["age"])
    out[MONEY_COLS] = out[MONEY_COLS].fillna(0)
    for col in CATEGORY_COLS:
        out[col] = out[col].fillna(stats[col])
    out[["cabin", "name"]] = out[["cabin", "name"]].fillna("Unknown")
    out[BOOL_COLS] = out[BOOL_COLS].astype(bool)
    return out


clean_stats = fit_cleaner(train_df)
train_clean = apply_cleaner(train_df, clean_stats)
val_clean = apply_cleaner(val_df, clean_stats)
test_clean = apply_cleaner(test_df, clean_stats)

print("Learned from train split:", clean_stats)
print("Missing after cleaning:",
      int(train_clean.isna().sum().sum()), int(val_clean.isna().sum().sum()),
      int(test_clean.isna().sum().sum()))

Learned from train split: {'age': np.float64(27.0), 'homeplanet': 'Earth', 'cryosleep': False, 'destination': 'TRAPPIST-1e', 'vip': False}
Missing after cleaning: 0 0 0


## Feature selection on the train split only
Keep features whose correlation / association with the target is above 0.2.

In [4]:
from dython.nominal import associations

fs = train_clean.drop(columns="passengerid").copy()
fs[target] = fs[target].astype(int)

num_features = fs.select_dtypes(include=[np.number]).columns.drop(target)
correlations = fs[num_features].corrwith(fs[target])
selected_num_features = correlations[abs(correlations) > 0.2].index.tolist()

cat_features = fs.select_dtypes(include=["object", "str", "bool", "category"]).columns
assoc = associations(fs[list(cat_features) + [target]], nominal_columns="all", plot=False)
target_assoc = assoc["corr"][target].drop(target)
selected_cat_features = target_assoc[target_assoc > 0.2].index.tolist()

selected_features = selected_num_features + selected_cat_features

print("Selected Numeric Features:", selected_num_features)
print("Selected Categorical Features:", selected_cat_features)
print(selected_features)

Selected Numeric Features: ['roomservice', 'spa', 'vrdeck']
Selected Categorical Features: ['cryosleep']
['roomservice', 'spa', 'vrdeck', 'cryosleep']


In [5]:
def to_model_input(df):
    """Pick the selected features; CatBoost wants categorical columns as strings."""
    X = df[selected_features].copy()
    X[selected_cat_features] = X[selected_cat_features].astype(str)
    return X


X_train_cb, X_val_cb, X_test_cb = (to_model_input(d) for d in (train_clean, val_clean, test_clean))
y_train, y_val, y_test = (d[target].astype(int) for d in (train_clean, val_clean, test_clean))

# this is use for prediction
print(X_train_cb.shape, X_val_cb.shape, X_test_cb.shape)

# this is use for answer
print(y_train.shape, y_val.shape, y_test.shape)

(5215, 4) (1739, 4) (1739, 4)
(5215,) (1739,) (1739,)


## Train CatBoost
Early stopping on validation AUC. `CATBOOST_PARAMS` is shared with `predict.ipynb`.

In [6]:
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

CATBOOST_PARAMS = dict(learning_rate=0.03, depth=5, l2_leaf_reg=5,
                       random_seed=42, allow_writing_files=False)

model = CatBoostClassifier(**CATBOOST_PARAMS, iterations=2000, eval_metric="AUC",
                           early_stopping_rounds=100, verbose=200)
model.fit(X_train_cb, y_train,
          cat_features=selected_cat_features,
          eval_set=(X_val_cb, y_val))

0:	test: 0.8017422	best: 0.8017422 (0)	total: 153ms	remaining: 5m 6s


200:	test: 0.8117609	best: 0.8119276 (152)	total: 502ms	remaining: 4.49s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.8119275703
bestIteration = 152

Shrink model to first 153 iterations.


CatBoostClassifier(allow_writing_files=False, depth=5, early_stopping_rounds=100, eval_metric='AUC', iterations=2000, l2_leaf_reg=5, learning_rate=0.03, random_seed=42, verbose=200)

In [7]:
def report(name, X, y):
    pred = model.predict(X).astype(int).ravel()
    proba = model.predict_proba(X)[:, 1]
    print(f"{name:10s} acc={accuracy_score(y, pred):.4f}  "
          f"f1={f1_score(y, pred):.4f}  auc={roc_auc_score(y, proba):.4f}")

report("train", X_train_cb, y_train)
report("validation", X_val_cb, y_val)

train      acc=0.7745  f1=0.7860  auc=0.8308
validation acc=0.7602  f1=0.7732  auc=0.8119


In [8]:
model.get_feature_importance(prettified=True)

,Feature Id,Importances
0,spa,36.792414
1,roomservice,26.816416
2,vrdeck,25.482886
3,cryosleep,10.908285


## Final evaluation on test (run once)

In [9]:
report("test", X_test_cb, y_test)

test       acc=0.7746  f1=0.7839  auc=0.8200
